In [2]:
# Example for running a PyTorch repo in Kaggle
!pip install -r requirements.txt
# Or install specific missing dependencies:
!pip install timm==0.4.12 opencv-python scikit-learn

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 6.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [3]:
import os
print("kvasir:", os.listdir("/kaggle/input/datasets/varshasrao/kvasirseg-og/Kvasir-SEG"))
print("clinicdb:", os.listdir("/kaggle/input/datasets/varshasrao/cvcclinicdb-og"))

kvasir: ['kavsir_bboxes.json', 'images', 'masks']
clinicdb: ['class_dict.csv', 'metadata.csv', 'TIF', 'PNG']


In [4]:
import os, sys, subprocess
from pathlib import Path

def run(cmd): subprocess.run(cmd, shell=True, check=True)

run("pip install -q gdown opencv-python-headless")

import torch, cv2, numpy as np, pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR   = Path("/kaggle/working")
INPUT_DIR  = Path("/kaggle/input")
REPO_DIR   = WORK_DIR / "PraNet-V2"
BINARY_DIR = REPO_DIR / "binary_seg"
TEST_SIZE  = 352
KVASIR_ROOT = INPUT_DIR / "mapped" / "PraNet" / "Kvasir"
CLINIC_ROOT = INPUT_DIR / "mapped" / "PraNet" / "CVC-ClinicDB"
WEIGHT_IDS = {
    "RES-V2.pth": "1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF",                          # PraNet-V2 checkpoint
    "res2net50_v1b_26w_4s-3cf99910.pth": "1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z",   # backbone, needed by model __init__
}

# --- Constants required by FlexiblePolypDataset (were missing) ---
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)  # ImageNet mean
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)  # ImageNet std

In [5]:
def clone_repo():
    if not REPO_DIR.exists():
        run(f"git clone --depth 1 https://github.com/ai4colonoscopy/PraNet-V2.git {REPO_DIR}")
    else:
        print("Repo already present, skipping clone.")

clone_repo()

Cloning into '/kaggle/working/PraNet-V2'...


In [6]:
import shutil
from pathlib import Path

# Create directory expected by relative path '../models' from /kaggle/working
target_dir = Path("/kaggle/models")
target_dir.mkdir(parents=True, exist_ok=True)

# Copy/symlink the downloaded backbone weight
src_weight = BINARY_DIR / "models" / "res2net50_v1b_26w_4s-3cf99910.pth"
dst_weight = target_dir / "res2net50_v1b_26w_4s-3cf99910.pth"

if src_weight.exists() and not dst_weight.exists():
    shutil.copy(src_weight, dst_weight)

In [7]:
import gdown
import shutil

def download_weights():
    snapshot_dir = BINARY_DIR / "snapshots" / "PraNet-V2"
    models_dir = BINARY_DIR / "models"
    
    # Create the parent directory '../../models' that Res2Net_v1b explicitly looks for
    alt_models_dir = REPO_DIR.parent / "models"
    
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)
    alt_models_dir.mkdir(parents=True, exist_ok=True)

    targets = {
        "RES-V2.pth": snapshot_dir / "RES-V2.pth",
        "res2net50_v1b_26w_4s-3cf99910.pth": models_dir / "res2net50_v1b_26w_4s-3cf99910.pth",
    }

    for fname, dest in targets.items():
        if not (dest.exists() and dest.stat().st_size > 1_000_000):
            gdown.download(id=WEIGHT_IDS[fname], output=str(dest), quiet=False)

    # Symlink or copy backbone weight to the parent path expected by Res2Net_v1b
    alt_backbone = alt_models_dir / "res2net50_v1b_26w_4s-3cf99910.pth"
    if not alt_backbone.exists():
        shutil.copy(targets["res2net50_v1b_26w_4s-3cf99910.pth"], alt_backbone)
    kaggle_models = Path("/kaggle/models")
    kaggle_models.mkdir(parents=True, exist_ok=True)
    shutil.copy(targets["res2net50_v1b_26w_4s-3cf99910.pth"], kaggle_models / "res2net50_v1b_26w_4s-3cf99910.pth")
    return targets["RES-V2.pth"]

CKPT_PATH = download_weights()
# Add this inside cell 4 right after downloading:


Downloading...
From (original): https://drive.google.com/uc?id=1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF
From (redirected): https://drive.google.com/uc?id=1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF&confirm=t&uuid=acbccb22-3e8e-44b4-bdbd-e905efc45570
To: /kaggle/working/PraNet-V2/binary_seg/snapshots/PraNet-V2/RES-V2.pth
100%|██████████| 131M/131M [00:00<00:00, 150MB/s]  
Downloading...
From: https://drive.google.com/uc?id=1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z
To: /kaggle/working/PraNet-V2/binary_seg/models/res2net50_v1b_26w_4s-3cf99910.pth
100%|██████████| 103M/103M [00:00<00:00, 149MB/s]  


In [8]:
import importlib
import sys
import os
import re
from pathlib import Path

def load_pranet_v2(ckpt_path: Path):
    if str(BINARY_DIR) not in sys.path:
        sys.path.insert(0, str(BINARY_DIR))
    if str(BINARY_DIR / "lib") not in sys.path:
        sys.path.insert(0, str(BINARY_DIR / "lib"))
    
    # Import PraNet model module directly
    try:
        from lib.pranet import PraNet_V2
        model = PraNet_V2(channel=32)
    except Exception:
        pranet_module = importlib.import_module("lib.pranet")
        # Fallback to PraNet if available
        model_cls = getattr(pranet_module, "PraNet", None) or getattr(pranet_module, "PraNet_V2")
        model = model_cls()

    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # Fix channel size mismatch on Conv and BatchNorm layers for binary segmentation
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.BatchNorm2d) and module.num_features == 3:
            weight_key = f"{name}.weight"
            if weight_key in state_dict and state_dict[weight_key].shape[0] == 1:
                parent_name, _, child_name = name.rpartition('.')
                parent = model if not parent_name else model.get_submodule(parent_name)
                setattr(parent, child_name, torch.nn.BatchNorm2d(1).to(DEVICE))
                
        elif isinstance(module, torch.nn.Conv2d) and module.out_channels == 3:
            weight_key = f"{name}.weight"
            if weight_key in state_dict and state_dict[weight_key].shape[0] == 1:
                new_conv = torch.nn.Conv2d(
                    in_channels=module.in_channels,
                    out_channels=1,
                    kernel_size=module.kernel_size,
                    stride=module.stride,
                    padding=module.padding,
                    bias=module.bias is not None
                )
                parent_name, _, child_name = name.rpartition('.')
                parent = model if not parent_name else model.get_submodule(parent_name)
                setattr(parent, child_name, new_conv)

    info = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded weights! Missing keys: {len(info.missing_keys)}, Unexpected: {len(info.unexpected_keys)}")
    
    return model.to(DEVICE).eval()

model = load_pranet_v2(CKPT_PATH)

Loaded weights! Missing keys: 0, Unexpected: 2


In [9]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
import torch.nn.functional as F

# --- Metrics Helper ---
def dice_iou(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    dice = (2.0 * intersection) / (pred.sum() + gt.sum() + 1e-8)
    iou = intersection / (union + 1e-8)
    return float(dice), float(iou)

# --- Fixed Dataset Class ---
class FlexiblePolypDataset(Dataset):
    def __init__(self, root: Path, testsize: int = TEST_SIZE):
        self.testsize = testsize
        self.pairs = []
        root = Path(root)

        # Common polyp dataset subfolder names
        valid_img_names = {"images", "original", "image", "png", "tif"}
        valid_mask_names = {"masks", "ground truth", "mask", "groundtruth"}

        img_dirs = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in valid_img_names]
        mask_dirs = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in valid_mask_names]

        # Fallback: if images and masks are directly in subfolders of root
        if not img_dirs:
            img_dirs = [root / "images"] if (root / "images").exists() else [root]
        if not mask_dirs:
            mask_dirs = [root / "masks"] if (root / "masks").exists() else [root]

        mask_map = {}
        for md in mask_dirs:
            for mp in md.iterdir():
                if mp.is_file() and mp.suffix.lower() in IMG_EXT:
                    mask_map[mp.stem] = mp

        for idir in img_dirs:
            for ip in idir.iterdir():
                if ip.is_file() and ip.suffix.lower() in IMG_EXT:
                    if ip.stem in mask_map:
                        self.pairs.append((ip, mask_map[ip.stem]))

        print(f"Found {len(self.pairs)} image-mask pairs in {root}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_h, gt_w = mask.shape[:2]

        resized = cv2.resize(image, (self.testsize, self.testsize), interpolation=cv2.INTER_LINEAR)
        tensor = (resized.astype(np.float32) / 255.0 - MEAN) / STD
        tensor = torch.from_numpy(tensor.transpose(2, 0, 1)).float()
        gt = (mask > 127).astype(np.float32)

        return tensor, torch.from_numpy(gt), (gt_h, gt_w), img_path.stem

# --- Per-Image Evaluation Function ---
@torch.no_grad()
def evaluate_dataset_per_image(model, dataset: Dataset, dataset_name: str, threshold: float = 0.5):
    records = []
    
    if len(dataset) == 0:
        print(f"Skipping {dataset_name}: No image pairs found.")
        return records

    print(f"Evaluating {len(dataset)} images for {dataset_name}...")
    for idx in range(len(dataset)):
        tensor, gt, (orig_h, orig_w), img_id = dataset[idx]
        tensor = tensor.unsqueeze(0).to(DEVICE)
        
        # PraNet forward pass (returns lateral maps; lateral_map_2 is index -1)
        outputs = model(tensor)
        res = outputs[-1] if isinstance(outputs, (tuple, list)) else outputs

        # Upsample to original mask resolution
        res = F.interpolate(res, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
        pred_prob = torch.sigmoid(res).squeeze().cpu().numpy()
        pred_mask = (pred_prob > threshold).astype(np.float32)
        gt_np = gt.numpy()

        dice, iou = dice_iou(pred_mask, gt_np)

        records.append({
            "dataset": dataset_name,
            "image_id": img_id,
            "dice": round(dice, 4),
            "iou": round(iou, 4)
        })

    return records

# --- Run Evaluation Across Datasets ---
all_image_metrics = []

datasets_to_eval = {
    "Kvasir-SEG": Path("/kaggle/input/datasets/varshasrao/kvasirseg-og/Kvasir-SEG"),
    "CVC-ClinicDB": Path("/kaggle/input/datasets/varshasrao/cvcclinicdb-og/PNG"),
}

for name, path in datasets_to_eval.items():
    if not path.exists():
        # Fallback to mapped directory if kaggle/input/datasets doesn't exist
        path = KVASIR_ROOT if "Kvasir" in name else CLINIC_ROOT
    
    dataset = FlexiblePolypDataset(path)
    metrics = evaluate_dataset_per_image(model, dataset, name)
    all_image_metrics.extend(metrics)

# --- Save and Inspect Per-Image Results ---
df_per_image = pd.DataFrame(all_image_metrics)

csv_save_path = WORK_DIR / "per_image_metrics.csv"
df_per_image.to_csv(csv_save_path, index=False)
print(f"\nPer-image metrics saved to: {csv_save_path}")

# Preview individual image scores
print("\nFirst 10 image results:")
print(df_per_image.head(10))

# Summary by dataset
print("\nDataset Summary Averages:")
print(df_per_image.groupby("dataset")[["dice", "iou"]].mean())

Found 1000 image-mask pairs in /kaggle/input/datasets/varshasrao/kvasirseg-og/Kvasir-SEG
Evaluating 1000 images for Kvasir-SEG...
Found 612 image-mask pairs in /kaggle/input/datasets/varshasrao/cvcclinicdb-og/PNG
Evaluating 612 images for CVC-ClinicDB...

Per-image metrics saved to: /kaggle/working/per_image_metrics.csv

First 10 image results:
      dataset                   image_id    dice     iou
0  Kvasir-SEG  cju7d1tvt25bu08019dvw3uff  0.0053  0.0027
1  Kvasir-SEG  cju30qbm1ad3x0855znuhpz9u  0.0084  0.0042
2  Kvasir-SEG  cju5xq3tdm9fn0987pbedxdg5  0.0271  0.0138
3  Kvasir-SEG  cju7deifq2fzn0755lc8idyh8  0.0053  0.0027
4  Kvasir-SEG  cju2raxlosl630988jdbfy9b0  0.0156  0.0078
5  Kvasir-SEG  cju5uget8krjy0818kvywd0zu  0.0022  0.0011
6  Kvasir-SEG  cju8b1v3br45u087189kku66u  0.0044  0.0022
7  Kvasir-SEG  cju1hs0za7jha0855vj0mdrjt  0.0018  0.0009
8  Kvasir-SEG  cju171py4qiha0835u8sl59ds  0.0082  0.0041
9  Kvasir-SEG  ck2bxiswtxuw80838qkisqjwz  0.0100  0.0050

Dataset Summary Averages: